In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import random
import numpy as np
import tensorflow as tf


# ============================================================
# Configuração
# ============================================================

SEED = 42
MODEL_OUT = "modified_tinyvgg_fashionmnist.h5"
EPOCHS = 20
BATCH_SIZE = 128
LEARNING_RATE = 1e-3

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)


# ============================================================
# Dataset
# ============================================================

def load_fashion_mnist():
    (x_train, y_train), (x_test, y_test) = tf.keras.datasets.fashion_mnist.load_data()

    # Normalização para [0,1]
    x_train = x_train.astype("float32") / 255.0
    x_test = x_test.astype("float32") / 255.0

    # Adiciona canal: (28,28) -> (28,28,1)
    x_train = np.expand_dims(x_train, axis=-1)
    x_test = np.expand_dims(x_test, axis=-1)

    return (x_train, y_train), (x_test, y_test)


# ============================================================
# Modelo: Modified-TinyVGG plausível
# 6 Conv + 4 FC
# ============================================================

def build_modified_tinyvgg(input_shape=(28, 28, 1), num_classes=10):
    model = tf.keras.Sequential(
        [
            tf.keras.layers.Input(shape=input_shape, name="input"),

            # Bloco 1
            tf.keras.layers.Conv2D(
                16, (3, 3), padding="same", activation="relu", name="conv1"
            ),
            tf.keras.layers.Conv2D(
                16, (3, 3), padding="same", activation="relu", name="conv2"
            ),
            tf.keras.layers.MaxPooling2D(pool_size=(2, 2), name="pool1"),

            # Bloco 2
            tf.keras.layers.Conv2D(
                32, (3, 3), padding="same", activation="relu", name="conv3"
            ),
            tf.keras.layers.Conv2D(
                32, (3, 3), padding="same", activation="relu", name="conv4"
            ),
            tf.keras.layers.MaxPooling2D(pool_size=(2, 2), name="pool2"),

            # Bloco 3
            tf.keras.layers.Conv2D(
                48, (3, 3), padding="same", activation="relu", name="conv5"
            ),
            tf.keras.layers.Conv2D(
                48, (3, 3), padding="same", activation="relu", name="conv6"
            ),
            tf.keras.layers.MaxPooling2D(pool_size=(2, 2), name="pool3"),

            tf.keras.layers.Flatten(name="flatten"),

            # 4 fully connected
            tf.keras.layers.Dense(128, activation="relu", name="fc1"),
            tf.keras.layers.Dense(64, activation="relu", name="fc2"),
            tf.keras.layers.Dense(32, activation="relu", name="fc3"),
            tf.keras.layers.Dense(num_classes, activation="softmax", name="fc4"),
        ],
        name="Modified_TinyVGG_FashionMNIST",
    )

    return model


# ============================================================
# Treinamento
# ============================================================

def main():
    (x_train, y_train), (x_test, y_test) = load_fashion_mnist()

    model = build_modified_tinyvgg()

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )

    model.summary()

    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_accuracy",
            patience=3,
            restore_best_weights=True,
            verbose=1,
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=2,
            min_lr=1e-6,
            verbose=1,
        ),
    ]

    history = model.fit(
        x_train,
        y_train,
        validation_split=0.1,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        shuffle=True,
        callbacks=callbacks,
        verbose=1,
    )

    test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
    print(f"\nTest loss: {test_loss:.6f}")
    print(f"Test accuracy: {test_acc:.6f}")

    # Salva em .h5
    model.save(MODEL_OUT)
    print(f"\nModelo salvo em: {MODEL_OUT}")

    return model, history


if __name__ == "__main__":
    main()

29515/29515 ━━━━━━━━━━━━━━━━━━━━ 0s 9us/step 
26421880/26421880 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step
5148/5148 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
4422102/4422102 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


Model: "Modified_TinyVGG_FashionMNIST"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1 (Conv2D)                  │ (None, 28, 28, 16)     │           160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2 (Conv2D)                  │ (None, 28, 28, 16)     │         2,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool1 (MaxPooling2D)            │ (None, 14, 14, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv3 (Conv2D)                  │ (None, 14, 14, 32)     │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv4 (Conv2D)                  │ (None, 14, 14, 32)     │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool2 (MaxPooling2D)            │ (None, 7, 7, 32)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv5 (Conv2D)                  │ (None, 7, 7, 48)       │        13,872 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv6 (Conv2D)                  │ (None, 7, 7, 48)       │        20,784 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool3 (MaxPooling2D)            │ (None, 3, 3, 48)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 432)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fc1 (Dense)                     │ (None, 128)            │        55,424 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fc2 (Dense)                     │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fc3 (Dense)                     │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fc4 (Dense)                     │ (None, 10)             │           330 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 117,114 (457.48 KB)

 Trainable params: 117,114 (457.48 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
422/422 ━━━━━━━━━━━━━━━━━━━━ 78s 167ms/step - accuracy: 0.7494 - loss: 0.6800 - val_accuracy: 0.8393 - val_loss: 0.4475 - learning_rate: 0.0010
Epoch 2/20
422/422 ━━━━━━━━━━━━━━━━━━━━ 60s 143ms/step - accuracy: 0.8550 - loss: 0.3945 - val_accuracy: 0.8665 - val_loss: 0.3639 - learning_rate: 0.0010
Epoch 3/20
422/422 ━━━━━━━━━━━━━━━━━━━━ 61s 145ms/step - accuracy: 0.8807 - loss: 0.3212 - val_accuracy: 0.8875 - val_loss: 0.3126 - learning_rate: 0.0010
Epoch 4/20
422/422 ━━━━━━━━━━━━━━━━━━━━ 48s 114ms/step - accuracy: 0.8961 - loss: 0.2807 - val_accuracy: 0.8953 - val_loss: 0.2934 - learning_rate: 0.0010
Epoch 5/20
422/422 ━━━━━━━━━━━━━━━━━━━━ 56s 133ms/step - accuracy: 0.9076 - loss: 0.2545 - val_accuracy: 0.9030 - val_loss: 0.2794 - learning_rate: 0.0010
Epoch 6/20
422/422 ━━━━━━━━━━━━━━━━━━━━ 64s 152ms/step - accuracy: 0.9143 - loss: 0.2332 - val_accuracy: 0.9080 - val_loss: 0.2694 - learning_rate: 0.0010
Epoch 7/20
422/422 ━━━━━━━━━━━━━━━━━━━━ 74s 133ms/step - accuracy: 0.9


Test loss: 0.254131
Test accuracy: 0.919600

Modelo salvo em: modified_tinyvgg_fashionmnist.h5


In [2]:
import numpy as np
import tensorflow as tf

NPZ_OUT = "fashion_mnist_full.npz"

# Carrega o Fashion-MNIST completo
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.fashion_mnist.load_data()

# Opcional: versões normalizadas com canal explícito
x_train_float32 = x_train.astype("float32") / 255.0
x_test_float32 = x_test.astype("float32") / 255.0

x_train_float32 = np.expand_dims(x_train_float32, axis=-1)  # (60000, 28, 28, 1)
x_test_float32 = np.expand_dims(x_test_float32, axis=-1)    # (10000, 28, 28, 1)

class_names = np.array([
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"
], dtype=object)

# Salva tudo em um único .npz
np.savez_compressed(
    NPZ_OUT,
    x_train_uint8=x_train,
    y_train=y_train,
    x_test_uint8=x_test,
    y_test=y_test,
    x_train_float32=x_train_float32,
    x_test_float32=x_test_float32,
    class_names=class_names,
)

print(f"Arquivo salvo em: {NPZ_OUT}")
print(f"x_train_uint8 shape: {x_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"x_test_uint8 shape: {x_test.shape}")
print(f"y_test shape: {y_test.shape}")

Arquivo salvo em: fashion_mnist_full.npz
x_train_uint8 shape: (60000, 28, 28)
y_train shape: (60000,)
x_test_uint8 shape: (10000, 28, 28)
y_test shape: (10000,)


In [13]:
from tflmfi import Experiment, load_config

cfg = load_config("configs/memoryfi_example.yaml")
exp = Experiment(cfg)

In [5]:
exp.quantize_model(
    model_path="models/modified_tinyvgg_fashionmnist.h5",
    dataset_name="fashion_mnist",           # carrega dados de calibração automaticamente
    output_path="models/modified_tinyvgg_fashionmnist_int8.tflite" # output_path defaults to cfg.model.tflite (e.g. models/lenet5_mnist_int8.tflite)
)

INFO:tensorflow:Assets written to: C:\Users\ronal\AppData\Local\Temp\tmpsoitlv8o\assets


INFO:tensorflow:Assets written to: C:\Users\ronal\AppData\Local\Temp\tmpsoitlv8o\assets


Saved artifact at 'C:\Users\ronal\AppData\Local\Temp\tmpsoitlv8o'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 28, 28, 1), dtype=tf.float32, name='input')
Output Type:
  TensorSpec(shape=(None, 10), dtype=tf.float32, name=None)
Captures:
  2047988741520: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2047988740368: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2047988739600: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2047994320784: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2047994322896: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2047994321168: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2047994323472: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2047994322320: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2047994323664: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2047994324048: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2047994324624

c:\Users\ronal\AppData\Local\Programs\Python\Python312\Lib\site-packages\tensorflow\lite\python\convert.py:846: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
[model_utils] Quantized model saved → models\modified_tinyvgg_fashionmnist_int8.tflite (138,736 bytes)
[experiment] Quantized model ready at models\modified_tinyvgg_fashionmnist_int8.tflite


WindowsPath('models/modified_tinyvgg_fashionmnist_int8.tflite')

In [11]:
exp.prepare_test_subset(
    subset_size=100,
    dataset_name="fashion_mnist",   # carrega test split automaticamente
    seed=42,
    output_path="models/fashion_mnist_100_subset.npz"
    # output_path defaults to cfg.model.subset_npz (e.g. models/mnist_subset.npz)
)

[model_utils] Test subset saved → models\fashion_mnist_100_subset.npz (100/10000 samples, seed=42)
[experiment] Test subset ready at models\fashion_mnist_100_subset.npz


WindowsPath('models/fashion_mnist_100_subset.npz')

In [14]:
exp.prepare()

[experiment] Preparing campaign 'memory' → C:\Downloads\framework_jose\runs\memoryfi_tinyVGG_fashion_minist_20260416_165233
[experiment] Installed 4 headers → C:\Downloads\framework_jose\firmware\src
[experiment] Manifest → C:\Downloads\framework_jose\runs\memoryfi_tinyVGG_fashion_minist_20260416_165233\run_manifest.json


CampaignArtifacts(campaign_type='memory', population_csv=WindowsPath('C:/Downloads/framework_jose/runs/memoryfi_tinyVGG_fashion_minist_20260416_165233/memory_population.csv'), selected_csv=WindowsPath('C:/Downloads/framework_jose/runs/memoryfi_tinyVGG_fashion_minist_20260416_165233/selected_faults.csv'), strata_csv=WindowsPath('C:/Downloads/framework_jose/runs/memoryfi_tinyVGG_fashion_minist_20260416_165233/selected_faults_by_layer.csv'), meta_json=WindowsPath('C:/Downloads/framework_jose/runs/memoryfi_tinyVGG_fashion_minist_20260416_165233/selected_faults.meta.json'), headers_dir=WindowsPath('C:/Downloads/framework_jose/runs/memoryfi_tinyVGG_fashion_minist_20260416_165233/headers'), all_csvs=[WindowsPath('C:/Downloads/framework_jose/runs/memoryfi_tinyVGG_fashion_minist_20260416_165233/memory_population.csv'), WindowsPath('C:/Downloads/framework_jose/runs/memoryfi_tinyVGG_fashion_minist_20260416_165233/selected_faults.csv'), WindowsPath('C:/Downloads/framework_jose/runs/memoryfi_tinyVG

In [15]:
exp.build()

[experiment] Building firmware (env=memory)...
[builder] Running: pio run -e memoryfi -d C:\Downloads\framework_jose\firmware
[builder] fault_duration=transient
[experiment] Build succeeded.


In [16]:
exp.flash()         # pio run -e registerfi -t upload


[experiment] Flashing via COM17 (env=memory)...
[builder] Running: pio run -e memoryfi -t upload -d C:\Downloads\framework_jose\firmware --upload-port COM17
[experiment] Flash succeeded.


In [ ]:
exp.run()

[experiment] Capturing UART → C:\Downloads\framework_jose\runs\memoryfi_tinyVGG_fashion_minist_20260416_165233\results.csv
[uart] Listening on COM17 @ 460800 baud
[uart] Waiting for boot event...
[uart] Boot received — campaign running.
[WARN] Non-JSON: Didn't find op for builtin opcode 'MAX_POOL_2D'
[WARN] Non-JSON: Failed to get registration from op code MAX_POOL_2D
[WARN] Non-JSON: E (1870) memoryfi: AllocateTensors failed.
[WARN] Non-JSON: I (1870) main_task: Returned from app_main()


In [11]:
exp.analyze() 

[experiment] Classifying C:\Downloads\framework_jose\runs\multi_memoryfi_lenet5_mnist_20260414_091416\results.csv...



Classification Summary  (100000 fault results)
  Masked         :  93369  ( 93.4%)
  Acceptable     :   2561  (  2.6%)
  Observed       :      0  (  0.0%)
  Critical       :   3479  (  3.5%)
  Crash          :      0  (  0.0%)
  Good           :      0  (  0.0%)
  NoInjection    :      0  (  0.0%)



[experiment] Classified → C:\Downloads\framework_jose\runs\multi_memoryfi_lenet5_mnist_20260414_091416\classified_results.csv


WindowsPath('C:/Downloads/framework_jose/runs/multi_memoryfi_lenet5_mnist_20260414_091416/classified_results.csv')